# GraphRAG: Fine-Grained Step-by-Step Pipeline

This notebook runs the complete MP2 experiment broken down into **isolated, independent steps**:

1. **Step 1 (Graphs):** Generate raw SBM knowledge graphs with matched-degree planted bridges and hubs.
2. **Step 2 (Communities):** Run Leiden community detection (L0, L1), edge stratification (within vs cut), and question generation.
3. **Step 3a (Prompt 1 - Default):** Run standard GraphRAG community summaries on GPU (~35 min for 40 graphs).
4. **Step 3b (Prompt 2 - Topology):** Run topology-aware prompts marking high-betweenness bridges.
5. **Step 3c (Prompt 3 - Control):** Run control prompts marking random nodes.
6. **Step 4 (Evaluate):** Fast deterministic evaluation of retention (L1/L2) and Oracle QA.
7. **Step 5 (Report):** Statistical aggregation, hypothesis tests (t-tests, Holm correction), and `summary.md`.
8. **Diagnostics (`inspect_run.py`):** Inspect raw graphs, community partitions, prompts, summaries, and QA failure cases.


In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Install zstd, Ollama, python client & start daemon
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

import subprocess, time, shutil, urllib.request
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server ready
print('Starting Ollama daemon...')
for _ in range(15):
    try:
        with urllib.request.urlopen('http://localhost:11434/') as resp:
            if resp.status == 200:
                print('Ollama server is up and running!')
                break
    except Exception:
        pass
    time.sleep(2)

# Pull Qwen3:8b
!ollama pull qwen3:8b

In [ ]:
# Cell 3: Clone or update repo & install dependencies
import os
if not os.path.exists('/kaggle/working/mp2'):
    !git clone https://github.com/Bvijay03/GraphRag.git /kaggle/working/mp2
%cd /kaggle/working/mp2
!git pull

!pip install -q -r requirements.txt

In [ ]:
# Cell 4: Verify Ollama is ready
from llm import OllamaLLM
llm = OllamaLLM(model='qwen3:8b', num_ctx=4096, temperature=0, seed=42, think=False)
res = llm.generate('Respond in one word: ready?')
print('Ollama LLM status:', res.text.strip())

### Step 1: Generate Knowledge Graphs (Takes ~1 second)
Generates the raw SBM graphs with planted bridges and hubs (matched degree) and saves to `runs/kaggle_exp/graphs_raw/`.

In [ ]:
# Cell 5: Step 1 (Generate Graphs)
!python step1_graph.py --config config_kaggle.yaml --out runs/kaggle_exp

### Step 2: Community Detection & Stratification (Takes ~2 seconds)
Runs Leiden hierarchical clustering, partitions triples into within-community vs cut strata, and prepares multi-hop questions. Saves to `runs/kaggle_exp/communities/`.

In [ ]:
# Cell 6: Step 2 (Leiden Communities)
!python step2_communities.py --config config_kaggle.yaml --out runs/kaggle_exp

### Step 3a: Summarization — Prompt 1: Default (GPU Accelerated, ~35 min)
Runs standard GraphRAG prompts for all communities. Saves raw summaries to `runs/kaggle_exp/summaries/default/`.
*Resumable: if interrupted, re-running automatically resumes where it left off.*

In [ ]:
# Cell 7: Step 3a (Prompt 1: Default)
!python step3_summarize.py --config config_kaggle.yaml --out runs/kaggle_exp --condition default --model qwen3:8b

### Step 3b: Summarization — Prompt 2: Topology (Optional / Next Day, ~35 min)
Prompts the LLM with topological marks pointing out high-betweenness bridge entities. Saves to `runs/kaggle_exp/summaries/topology/`.

In [ ]:
# Cell 8: Step 3b (Prompt 2: Topology)
!python step3_summarize.py --config config_kaggle.yaml --out runs/kaggle_exp --condition topology --model qwen3:8b

### Step 3c: Summarization — Prompt 3: Control (Optional / Next Day, ~35 min)
Prompts the LLM with random entity marks to control for attention priming. Saves to `runs/kaggle_exp/summaries/control/`.

In [ ]:
# Cell 9: Step 3c (Prompt 3: Control)
!python step3_summarize.py --config config_kaggle.yaml --out runs/kaggle_exp --condition control --model qwen3:8b

### Step 4: Evaluate Retention & QA (Takes ~2 seconds)
Evaluates whichever prompt conditions have been completed so far. Computes L1/L2 retention, within vs cut stratum loss, and exact Oracle QA accuracy.

In [ ]:
# Cell 10: Step 4 (Evaluate)
!python step4_evaluate.py --config config_kaggle.yaml --out runs/kaggle_exp

### Step 5: Statistical Reporting (Takes ~1 second)
Aggregates statistics across graphs, runs t-tests and Holm correction, and outputs `summary.md` and `results.json`.

In [ ]:
# Cell 11: Step 5 (Report)
!python step5_report.py --config config_kaggle.yaml --out runs/kaggle_exp

from IPython.display import Markdown, display
display(Markdown(filename='runs/kaggle_exp/summary.md'))

### Diagnostic Inspector: Inspect Prompts, Summaries & QA Errors
Inspect individual graphs, community prompts, or diagnose wrong answers.

In [ ]:
# Cell 12: Run overview of completed stages
!python inspect_run.py --dir runs/kaggle_exp

# Inspect raw graph 1 (nodes, degree, betweenness):
# !python inspect_run.py --dir runs/kaggle_exp --seed 1 --raw

# Inspect Prompt & Summary for Community 0 under default prompt:
# !python inspect_run.py --dir runs/kaggle_exp --seed 1 --community 0 --condition default

# Inspect QA failures:
# !python inspect_run.py --dir runs/kaggle_exp --seed 1 --qa-errors

### Download Results
Zip the outputs for downloading to your local machine.

In [ ]:
# Cell 13: Zip results
!zip -r /kaggle/working/mp2_results.zip runs/kaggle_exp/
print('Download available at /kaggle/working/mp2_results.zip')